# ACE-SQL benchmark-overlap audit

## tl;dr

This notebook checks question, SQL, schema, and database-identifier overlap separately. It requires cross-level evidence before calling a match contamination: a generic question or `COUNT(*)` query on an unrelated schema is only a lexical collision. The notebook displays the audit results; the CLI writes path-sanitized ledgers under `audits/results/`.

## Context & Methods

The intended use is to assess whether the 7,092-example SFT roles and the 2,913-example RL split overlap BIRD Dev or Spider Dev. Normalized question matching preserves entities and numbers. SQL matching preserves literals. Schema matching uses normalized user-defined `table.column` identifiers and excludes SQLite's internal `sqlite_*` tables. Near-question retrieval uses character 3-5 gram TF-IDF cosine together with token Jaccard; schema-column similarity is Jaccard, while a separate structural fingerprint retains coarse types, primary keys, and foreign keys. Database-content equality remains unresolved when the SynSQL SQLite files are absent.

### Key assumptions

- The SFT retriever file is read from the worktree when supplied, otherwise from the fixed Git `HEAD` blob.
- The generator prompt schema is reported as a prompt schema, not silently treated as a full catalog.
- Exact or near SQL alone is not labeled leakage without database/schema agreement.

In [1]:
from pathlib import Path
import os
import json
import pandas as pd
from IPython.display import display

AUDIT_DIR = Path.cwd().resolve()
if not (AUDIT_DIR / 'benchmark_overlap_audit.py').exists():
    AUDIT_DIR = (AUDIT_DIR / 'audits').resolve()
REPO = AUDIT_DIR.parent
RESULT_DIR = AUDIT_DIR / 'results'

def required_input(name):
    value = os.environ.get(name)
    if not value:
        raise RuntimeError(f'Set {name} to the corresponding benchmark JSON file')
    return Path(value)

BIRD_JSON = required_input('ACE_SQL_AUDIT_BIRD_JSON')
BIRD_TABLES = required_input('ACE_SQL_AUDIT_BIRD_TABLES')
SPIDER_JSON = required_input('ACE_SQL_AUDIT_SPIDER_JSON')
SPIDER_TABLES = required_input('ACE_SQL_AUDIT_SPIDER_TABLES')
retriever_value = os.environ.get('ACE_SQL_AUDIT_RETRIEVER_SFT_JSON')
RETRIEVER_SFT = Path(retriever_value) if retriever_value else None
RESULT_DIR.mkdir(parents=True, exist_ok=True)

## Data

In [2]:
import sys
sys.path.insert(0, str(AUDIT_DIR))
from benchmark_overlap_audit import run_audit
from argparse import Namespace

args = Namespace(
    repo=REPO,
    retriever_sft_json=RETRIEVER_SFT,
    bird_json=BIRD_JSON,
    bird_tables=BIRD_TABLES,
    spider_json=SPIDER_JSON,
    spider_tables=SPIDER_TABLES,
    output_dir=RESULT_DIR,
)
results = run_audit(args)
display(pd.DataFrame(results['profiles']))
display(pd.DataFrame(results['manifest']['inputs']).T)

,source,rows,unique_normalized_questions,unique_normalized_sql
0,SFT-retriever,7092,7088.0,NaN
1,SFT-generator,7092,7090.0,NaN
2,RL,2913,2913.0,NaN
3,SFT-generator SQL,7092,NaN,7072.0
4,RL SQL,2913,NaN,2913.0


,path,sha256,rows,origin
sft_generator,SFT/data/domain2_alpaca_think.json,81aaa694f0bc242a56d3653a2922bda7f9cfa97db292c8...,7092,NaN
sft_retriever,NaN,4bf2336ab25c5080a254bda6ef853f75746136c5ecb37f...,7092,git:HEAD:SFT/data/domain1_alpaca_think.json
rl_train,RL/data/train.parquet,1a91d90ebf1277bd3ec2bef8d3437629ed5ba805bbbd1f...,2913,NaN
bird_json,external:BIRD/dev.json,630272f2b1c44d8cef2c3b246f623355cf0bbc1e832c81...,1534,NaN
bird_tables,external:BIRD/dev_tables.json,35e7f73f2bc3a3bc4def9466215f7617ff8b37e9b2837f...,NaN,NaN
spider_json,external:Spider/dev.json,30d64a3fccde493226df79687aed9e4a1c0129525baf44...,1034,NaN
spider_tables,external:Spider/tables.json,61bb20aa401f03164e2d7f3b16509b7b5f79cc9c943ca7...,NaN,NaN


## Results

In [3]:
display(pd.DataFrame(results['question_summary']))
display(pd.DataFrame(results['sql_summary']))
display(pd.DataFrame(results['schema_summary']))
display(pd.DataFrame(results['db_id_summary']))

,benchmark,training_source,normalized_exact,char_tfidf_max,char_ge_0_85,char_ge_0_90,char_ge_0_95,high_confidence_near
0,BIRD,SFT-retriever,0,0.650092,0,0,0,0
1,BIRD,SFT-generator,0,0.852271,1,0,0,0
2,BIRD,RL,0,0.653113,0,0,0,0
3,Spider,SFT-retriever,0,0.879898,1,0,0,0
4,Spider,SFT-generator,0,0.858446,1,0,0,0
5,Spider,RL,0,0.706444,0,0,0,0


,benchmark,training_source,normalized_exact_rows,template_exact_rows
0,BIRD,SFT-generator,0,0
1,BIRD,RL,0,0
2,Spider,SFT-generator,2,2
3,Spider,RL,0,0


,benchmark,training_source,exact_column_catalog_dbs,near_column_catalog_dbs_ge_0_85,near_column_catalog_dbs_ge_0_95,exact_structural_schema_dbs
0,BIRD,SFT-retriever-full,0,0,0,NaN
1,BIRD,SFT-generator-prompt,0,0,0,NaN
2,BIRD,RL-full,0,0,0,0.0
3,Spider,SFT-retriever-full,0,0,0,NaN
4,Spider,SFT-generator-prompt,0,0,0,NaN
5,Spider,RL-full,1,1,1,0.0


,benchmark,rl_db_id_exact,rl_unique_db_ids,benchmark_unique_db_ids
0,BIRD,0,2232,11
1,Spider,0,2232,20


### Inspect all exact SQL and near-schema candidates

In [4]:
sql_candidates = pd.DataFrame(results['sql_exact_candidates'])
schema_candidates = pd.DataFrame(results['schema_candidates'])
display(sql_candidates)
display(schema_candidates)

,benchmark,training_source,benchmark_row,benchmark_db_id,benchmark_question,benchmark_sql,train_row,train_question,train_sql
0,Spider,SFT-generator,429,wta_1,Find the total number of players.,SELECT count(*) FROM players,6572,How many players are there in the players table?,SELECT COUNT(*) \nFROM players;
1,Spider,SFT-generator,430,wta_1,How many players are there?,SELECT count(*) FROM players,6572,How many players are there in the players table?,SELECT COUNT(*) \nFROM players;


,benchmark,training_source,benchmark_db_id,best_jaccard,training_rows,matched_training_row_count,training_db_ids,catalog_column_count,benchmark_db_question_count,exact_structural_schema,benchmark_only_columns,training_only_columns
0,Spider,RL-full,world_1,1.0,"[161, 281, 1130, 1205, 1640, 1794, 2214]",7,[world],24,120,False,[],[]


### Highest question-similarity candidates

In [5]:
top_questions = pd.DataFrame(results['question_top_matches'])
display(top_questions.groupby(['benchmark', 'training_source'], group_keys=False).head(3))

,benchmark,training_source,benchmark_row,train_row,char_tfidf_similarity,token_jaccard,high_confidence_near,benchmark_question,train_question
0,BIRD,SFT-retriever,892,4283,0.650092,0.136364,False,State the driver with the most points scored. ...,Drivers who scored more points than the averag...
1,BIRD,SFT-retriever,1278,6785,0.638456,0.161290,False,"Of the patients with an normal level of IGG, h...",Could you tell me the names of the hospitals a...
2,BIRD,SFT-retriever,1406,796,0.587317,0.142857,False,"Among the budgets for Food, which one has the ...","Return the IDs, names, and total budgets of th..."
5,BIRD,SFT-generator,1056,5983,0.852271,0.750000,False,What is the name of the football league in the...,Netherlands is the name of country;\nWhat's th...
6,BIRD,SFT-generator,1117,2957,0.758448,0.300000,False,What are the player api id of 10 heaviest play...,5 heaviest players refer to MAX(weight) limit ...
7,BIRD,SFT-generator,1079,221,0.706358,0.333333,False,Which player is the tallest?,tallest player refers to MAX(height_in_cm);\nW...
10,BIRD,RL,732,60,0.653113,0.230769,False,Which publisher published the slowest superhero?,What are the names of the publisher that publi...
11,BIRD,RL,1128,2504,0.649500,0.166667,False,Which country's players have the heaviest aver...,Who has the heaviest weight?
12,BIRD,RL,1144,2504,0.605994,0.357143,False,Please state the finishing rate and curve scor...,Who has the heaviest weight?
15,Spider,SFT-retriever,430,1346,0.879898,0.625000,False,How many players are there?,How many players are there in the players table?


## Takeaways

Interpret the executed tables conservatively. BIRD is clean under the tested exact and high-threshold near-overlap rules when no schema or database match appears. Spider requires a schema-level caveat if the RL `world` catalog matches `world_1`; without the SynSQL SQLite file, database-content identity cannot be ruled in or out. Any generic exact SQL collision on a different schema should be disclosed as a lexical collision, not promoted to contamination. This audit covers the retained 7,092-example-per-role SFT snapshots and 2,913-row RL split; it is not a substitute for a source-manifest audit of all SynSQL-2.5M records.